In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [2]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 6 teams with confirmed lineups


### Load Model

In [2]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 4.5


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Kristaps Porzingis,Over,18.5,-137,2025-11-26,2025-11-25T18:30:01Z
1,Underdog,player_points,Kristaps Porzingis,Under,18.5,-137,2025-11-26,2025-11-25T18:30:01Z
2,Underdog,player_points,Jalen Johnson,Over,23.5,-137,2025-11-26,2025-11-25T18:30:01Z
3,Underdog,player_points,Jalen Johnson,Under,23.5,-137,2025-11-26,2025-11-25T18:30:01Z
4,Underdog,player_points,C.J. McCollum,Over,17.5,-137,2025-11-26,2025-11-25T18:30:01Z


## Top EVs for 2 leg bets

### Underdog picks

In [4]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 31 players...
Processing 29 players...
Generated 337 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
221,Tyrese Maxey,LeBron James,32.5,20.5,-162,-130,26.23,15.88,0.781,0.789,under,under,1,81.15,0.406,High,Med
260,Anthony Black,Rui Hachimura,12.5,11.5,-125,-110,14.09,14.58,0.593,0.683,over,over,0,19.21,0.096,High,High
274,Trendon Watford,Brook Lopez,9.5,5.5,-130,-130,8.48,8.07,0.570,0.672,under,over,0,12.66,0.063,Med,Med
62,Zaccharie Risacher,Ivica Zubac,11.5,16.5,-110,-125,12.60,18.89,0.561,0.635,over,over,0,4.81,0.024,High,High
107,Nickeil Alexander-Walker,Austin Reaves,18.5,22.5,-115,-113,19.61,24.98,0.559,0.627,over,over,0,3.02,0.015,High,High
93,Bilal Coulibaly,Kris Dunn,10.5,6.5,-132,-126,9.66,8.16,0.556,0.620,under,over,0,1.47,0.007,Med,Med
242,Quentin Grimes,Marcus Smart,17.5,6.5,-152,-112,16.47,8.19,0.554,0.610,under,over,0,-0.62,0.000,High,High
9,Kristaps Porziņģis,Kawhi Leonard,18.5,20.5,-115,-125,17.54,22.50,0.554,0.605,under,over,0,-1.49,0.000,High,High
207,Jalen Suggs,James Harden,16.5,24.5,-118,-115,15.85,25.99,0.542,0.590,under,over,0,-6.01,0.000,High,High
162,Marvin Bagley III,Jaxson Hayes,7.5,7.5,-105,-122,8.01,6.68,0.539,0.571,over,under,0,-9.56,0.000,Med,Low


### Prizepicks picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 43 players...
Processing 39 players...
Generated 631 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
483,Jared McCain,LeBron James,13.5,20.5,-115,-130,9.13,15.88,under,under,0.785,0.789,0.6074,0.235,0.239,0.317,82.22,0.411,1,5.52,5.75,Med,Med,"(0.0, 20.0)","(4.6, 27.2)",0.05,0,82.2
395,Tyrese Maxey,Jake LaRavia,32.5,7.5,-162,-110,26.23,10.83,under,over,0.781,0.694,0.5313,0.210,0.123,0.216,59.38,0.297,0,8.09,6.55,High,High,"(10.4, 42.1)","(0.0, 23.7)",0.05,0,59.4
120,Alex Sarr,Rui Hachimura,17.5,11.5,-130,-110,20.33,14.58,over,over,0.652,0.683,0.4364,0.107,0.139,0.149,30.92,0.155,0,7.27,6.45,High,High,"(6.1, 34.6)","(1.9, 27.2)",0.05,0,30.9
527,Tristan da Silva,Brook Lopez,12.5,5.5,-120,-130,14.88,8.07,over,over,0.632,0.672,0.4159,0.076,0.116,0.116,24.77,0.124,0,7.08,5.79,High,Med,"(1.0, 28.8)","(0.0, 19.4)",0.05,0,24.8
566,Goga Bitadze,John Collins,5.5,11.5,-109,-130,6.74,13.91,over,over,0.605,0.643,0.3815,0.062,0.100,0.094,14.45,0.072,0,4.63,6.59,Low,High,"(0.0, 15.8)","(1.0, 26.8)",0.05,0,14.4
501,Anthony Black,Ivica Zubac,12.5,16.5,-125,-125,14.09,18.89,over,over,0.593,0.635,0.3695,0.038,0.080,0.068,10.86,0.054,0,6.74,6.89,High,High,"(0.9, 27.3)","(5.4, 32.4)",0.05,0,10.9
531,Andre Drummond,Austin Reaves,11.5,22.5,-123,-113,10.35,24.98,under,over,0.572,0.627,0.3513,0.031,0.086,0.066,5.38,0.027,0,6.32,7.67,High,High,"(0.0, 22.7)","(9.9, 40.0)",0.05,0,5.4
558,Trendon Watford,Kris Dunn,9.5,6.5,-130,-126,8.48,8.16,under,over,0.570,0.620,0.3468,0.009,0.059,0.039,4.03,0.020,0,5.77,5.43,Med,Med,"(0.0, 19.8)","(0.0, 18.8)",0.05,0,4.0
273,Zaccharie Risacher,Marcus Smart,11.5,6.5,-110,-112,12.60,8.19,over,over,0.561,0.610,0.3352,0.035,0.084,0.065,0.57,0.003,0,7.19,6.05,High,High,"(0.0, 26.7)","(0.0, 20.0)",0.05,0,0.6
87,Nickeil Alexander-Walker,Kawhi Leonard,18.5,20.5,-115,-125,19.61,22.50,over,over,0.559,0.605,0.3316,0.014,0.060,0.041,-0.53,0.000,0,7.48,7.49,High,High,"(5.0, 34.3)","(7.8, 37.2)",0.05,0,-0.5


## 3 leg parlay

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 31 players...
Processing 29 players...
Generated 3129 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
2697,Tyrese Maxey,LeBron James,Rui Hachimura,32.5,20.5,11.5,26.23,15.88,14.58,0.781,0.789,0.683,under,under,over,0,127.37,0.255,High,Med,High
2981,Anthony Black,Ivica Zubac,Brook Lopez,12.5,16.5,5.5,14.09,18.89,8.07,0.593,0.635,0.672,over,over,over,0,36.77,0.074,High,High,Med
3066,Trendon Watford,Austin Reaves,Marcus Smart,9.5,22.5,6.5,8.48,24.98,8.19,0.570,0.627,0.610,under,over,over,0,17.72,0.035,Med,High,High
929,Zaccharie Risacher,Kawhi Leonard,Kris Dunn,11.5,20.5,6.5,12.60,22.50,8.16,0.561,0.605,0.620,over,over,over,0,13.71,0.027,High,High,Med
1448,Nickeil Alexander-Walker,Quentin Grimes,James Harden,18.5,17.5,24.5,19.61,16.47,25.99,0.559,0.554,0.590,over,under,over,0,-1.23,0.000,High,High,High


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 43 players...
Processing 39 players...
Generated 8233 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
6317,Tyrese Maxey,Jared McCain,LeBron James,32.5,13.5,20.5,26.23,9.13,15.88,0.781,0.785,0.789,under,under,under,1,161.34,0.323,High,Med,Med
2365,Alex Sarr,Rui Hachimura,Jake LaRavia,17.5,11.5,7.5,20.33,14.58,10.83,0.652,0.683,0.694,over,over,over,0,66.95,0.134,High,High,High
7844,Tristan da Silva,John Collins,Brook Lopez,12.5,11.5,5.5,14.88,13.91,8.07,0.632,0.643,0.672,over,over,over,0,47.34,0.095,High,High,Med
8195,Goga Bitadze,Ivica Zubac,Kris Dunn,5.5,16.5,6.5,6.74,18.89,8.16,0.605,0.635,0.620,over,over,over,0,28.87,0.058,Low,High,Med
7654,Anthony Black,Austin Reaves,Marcus Smart,12.5,22.5,6.5,14.09,24.98,8.19,0.593,0.627,0.610,over,over,over,0,22.44,0.045,High,High,High


In [18]:
# df = playerScoring('Trey Murphy III', s26, current_date, teamStarPlayer, projectedStartingFive)
# playerContext('Trey Murphy III', s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)